# CSBP711 A1 — run the full experiment on Colab

Runtime → Change runtime type → **GPU (T4)**. Run the cells top to bottom (≈ 1 h per seed). Every cell uses absolute paths, so re-running one is safe.

## 0. Get the code — pick ONE of the two cells
**A.** Upload `cxr-algo-compare.zip` (the repository zip).  **B.** Clone the group's GitHub repo once it exists.

In [ ]:
# A) upload the zip
import os
from google.colab import files
os.chdir('/content')
!rm -rf /content/repo /content/cxr-algo-compare
up = files.upload()                      # choose cxr-algo-compare.zip
!unzip -q /content/cxr-algo-compare.zip -d /content && mv /content/cxr-algo-compare /content/repo
os.chdir('/content/repo')
!pip install -q -r requirements.txt
!nvidia-smi -L && ls

In [ ]:
# B) clone from GitHub (skip if you used A)
import os
REPO_URL = 'https://github.com/<org>/<repo>.git'   # <- the group repository
os.chdir('/content')
!rm -rf /content/repo
!git clone $REPO_URL /content/repo
os.chdir('/content/repo')
!pip install -q -r requirements.txt
!nvidia-smi -L && ls

## 1. Dataset (Kaggle mirror of Kermany et al. 2018, CC BY 4.0)
Kaggle → Settings → API. Either **Generate New Token** (a string shown on screen — paste it below), or **Legacy API Credentials → Create New Token** (downloads `kaggle.json` — use the second cell). If the download returns 403, open the dataset page once in your browser and re-run.

In [ ]:
# Route 1: token string
import os
os.chdir('/content/repo')
os.environ['KAGGLE_API_TOKEN'] = 'paste-your-token-here'
!pip install -q -U kaggle
!python scripts/download_data.py kaggle
!cat data/raw/DOWNLOAD_INFO.json

In [ ]:
# Route 2: legacy kaggle.json file (skip if Route 1 worked)
import os
from google.colab import files
os.chdir('/content/repo')
os.makedirs('/root/.kaggle', exist_ok=True)
up = files.upload()                      # choose kaggle.json
open('/root/.kaggle/kaggle.json','wb').write(up['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!python scripts/download_data.py kaggle
!cat data/raw/DOWNLOAD_INFO.json

## 2. Audit + split + cache, five models, both ablations, report
Set `SEEDS="42 43 44"` for mean ± std (3× the time). If the session disconnects, just re-run this cell: finished runs are skipped.

In [ ]:
import os; os.chdir('/content/repo')
# main run (seed 42 is done and committed; teammates use 43 / 44). FRACTIONS="" skips the data-size ablation.
!SEEDS="43" EPOCHS=12 FRACTIONS="" bash scripts/run_all.sh
# extra check (optional, ~25 min): same models with squashed instead of padded images
# !RESIZE=stretch SEEDS="42" EPOCHS=12 FRACTIONS="" ABLATIONS="" bash scripts/run_all.sh

## 3. Look at the results

In [ ]:
import os; os.chdir('/content/repo')
from IPython.display import Markdown, Image, display
display(Markdown(open('results/summary.md').read()))
for f in ['results/figures/main_auroc.png','results/figures/ablation_permute.png','results/figures/ablation_fraction.png']:
    if os.path.exists(f): display(Image(f))

## 4. Save the artefacts (results JSONs, tables, figures, audit) — send these back and commit them to the repo

In [ ]:
import os; os.chdir('/content/repo')
from google.colab import files
!zip -qr /content/a1_results.zip results data/audit.md data/audit.json data/index.csv data/raw/DOWNLOAD_INFO.json
files.download('/content/a1_results.zip')